In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: /Users/sushank/Documents/GitHub/urbanAi
src exists: True


# UrbanFlow AI - Phase 2 Preprocessing Notebook

This notebook links Phase 1 inspection with Phase 2 leakage-safe preprocessing for METR-LA.
It avoids modifying raw data and writes processed artifacts under `data/processed/METR-LA/`.

In [2]:
from __future__ import annotations

import json
import pickle
from pathlib import Path

import h5py
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data.preprocessing import (
    PreprocessingConfig,
    build_metadata,
    chronological_split,
    fit_scaler,
    load_metr_la_h5,
    save_config,
    save_json,
    scaler_to_metadata,
    transform_split,
    verify_array_health,
)
from src.data.sequence import generate_sequences, verify_sequence_shapes

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1) Environment and Project Paths

In [3]:
PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "data"
RAW_DIR = DATA_ROOT / "raw" / "METR-LA"
PROCESSED_DIR = DATA_ROOT / "processed" / "METR-LA"
CONFIG_PATH = PROJECT_ROOT / "configs" / "preprocessing_config.json"
TRAFFIC_FILE = RAW_DIR / "METR-LA.h5"
ADJ_FILE = RAW_DIR / "adj_METR-LA.pkl"

for p in [TRAFFIC_FILE, ADJ_FILE, CONFIG_PATH]:
    print(p, "exists=" , p.exists())

/Users/sushank/Documents/GitHub/urbanAi/data/raw/METR-LA/METR-LA.h5 exists= True
/Users/sushank/Documents/GitHub/urbanAi/data/raw/METR-LA/adj_METR-LA.pkl exists= True
/Users/sushank/Documents/GitHub/urbanAi/configs/preprocessing_config.json exists= True


## 2) Load and Inspect HDF5 Structure

In [4]:
with h5py.File(TRAFFIC_FILE, "r") as f:
    print("Root keys:", list(f.keys()))
    df_group = f["df"]
    print("df keys:", list(df_group.keys()))
    for key in ["axis0", "axis1", "block0_items", "block0_values"]:
        ds = df_group[key]
        print(f"{key}: shape={ds.shape}, dtype={ds.dtype}")

Root keys: ['df']
df keys: ['axis0', 'axis1', 'block0_items', 'block0_values']
axis0: shape=(207,), dtype=|S6
axis1: shape=(34272,), dtype=int64
block0_items: shape=(207,), dtype=|S6
block0_values: shape=(34272, 207), dtype=float64


## 3) Reconstruct `traffic_df` and Validate DataFrame Schema

In [5]:
traffic_df = load_metr_la_h5(TRAFFIC_FILE)

schema_summary = {
    "shape": traffic_df.shape,
    "index_type": str(type(traffic_df.index)),
    "column_count": int(traffic_df.shape[1]),
    "start": traffic_df.index.min().isoformat(),
    "end": traffic_df.index.max().isoformat(),
    "sampling_interval_mode": str(traffic_df.index.to_series().diff().mode().iloc[0]),
    "dtype_set": sorted({str(x) for x in traffic_df.dtypes}),
}
pd.Series(schema_summary)

shape                                       (34272, 207)
index_type                <class 'pandas.DatetimeIndex'>
column_count                                         207
start                                2012-03-01T00:00:00
end                                  2012-06-27T23:55:00
sampling_interval_mode                   0 days 00:05:00
dtype_set                                      [float64]
dtype: object